# 1. ARVO + Juliet Router 학습

이 노트북은 ARVO의 실제 취약점과 Juliet의 E3/E4/E6 CWE 사례를 결합해 Anchor/Rare Router를 학습합니다. Juliet 변환, 누수 없는 분할 병합, 정적 분석, 학습과 dev 보정을 순서대로 실행합니다. 이 과정은 LLM API를 호출하지 않습니다.

In [18]:
import hashlib
import json
import os
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

JULIET_SOURCE = Path(os.getenv(
    'JULIET_SOURCE_DIR',
    r'C:\Users\junhyun111\Downloads\2017-10-01-juliet-test-suite-for-c-cplusplus-v1-3',
)).expanduser()
ARVO_FEATURE_DIR = ROOT / 'data' / 'phase2e' / 'semantic'
JULIET_DIR = ROOT / 'data' / 'juliet'
JULIET_FEATURE_DIR = ROOT / 'data' / 'phase2e_juliet'
DATA_DIR = ROOT / 'data' / 'phase2e_combined'
JULIET_CONVERSION_CACHE = JULIET_DIR / '.conversion_cache.json'
JULIET_FEATURE_CACHE = JULIET_FEATURE_DIR / '.analysis_cache.json'
ARTIFACT_DIR = ROOT / 'artifacts' / 'phase2e'
ANCHOR_MODEL_PATH = ARTIFACT_DIR / 'router_anchor_rare_v2.pkl'
UTILITY_MODEL_PATH = ARTIFACT_DIR / 'router_top2_full5_v4.pkl'
SUMMARY_PATH = ARTIFACT_DIR / 'router_training_summary.json'
SEED = 2026
TARGET_RARE_RECALL = 0.95
JULIET_CASES_PER_FAMILY = 100
JULIET_CASES_PER_CWE = 50
JULIET_CASES_PER_TEMPLATE = 10
REBUILD_JULIET = False
REBUILD_JULIET_FEATURES = False
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def _read_cache(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def _source_fingerprint(paths):
    digest = hashlib.sha256()
    for path in sorted(paths):
        digest.update(str(path.relative_to(ROOT)).encode('utf-8'))
        digest.update(path.read_bytes())
    return digest.hexdigest()

CONVERSION_FINGERPRINT = _source_fingerprint([
    ROOT / 'src' / 'llm_security' / 'benchmarks' / 'juliet.py',
    ROOT / 'src' / 'llm_security' / 'cwe.py',
])
ANALYSIS_FINGERPRINT = _source_fingerprint([
    *sorted((ROOT / 'src' / 'llm_security' / 'analysis').glob('*.py')),
    ROOT / 'src' / 'llm_security' / 'cwe.py',
    ROOT / 'src' / 'llm_security' / 'experiments' / 'dataset.py',
])

from llm_security.benchmarks import prepare_juliet_dataset, merge_router_split_directories
from llm_security.datasets import load_router_samples_jsonl, load_utility_samples_jsonl
from llm_security.experiments import Phase2EConfig, prepare_phase2e_frozen_jsonl
from llm_security.models import ExpertFamily, to_dict
from llm_security.routing import (
    AnchorRareRouter,
    BudgetedUtilityRouter,
    UtilityPolicyConfig,
    assert_project_disjoint,
    split_gate_calibration_samples,
)

print('Juliet:', JULIET_SOURCE)
print('Combined features:', DATA_DIR)

Juliet: C:\Users\junhyun111\Downloads\2017-10-01-juliet-test-suite-for-c-cplusplus-v1-3
Combined features: C:\Users\junhyun111\Desktop\llm-security\data\phase2e_combined


## 데이터 준비

Juliet는 family/CWE/template별 상한을 둬 특정 복제 패턴이 학습 데이터를 지배하지 않도록 합니다. 번호만 다른 Juliet flow variant는 같은 프로젝트로 묶이므로 train/dev/test 사이에 가까운 복제본이 섞이지 않습니다. ARVO의 기존 프로젝트 분할도 그대로 보존됩니다.

In [19]:
required_splits = ('train', 'dev', 'test')
if not JULIET_SOURCE.is_dir():
    raise FileNotFoundError(f'Juliet 폴더를 찾을 수 없습니다: {JULIET_SOURCE}')
for split in required_splits:
    split_path = ARVO_FEATURE_DIR / f'router_{split}.jsonl'
    if not split_path.is_file():
        raise FileNotFoundError(
            f'기존 ARVO Router 데이터가 없습니다: {split_path}. '
            '먼저 ARVO phase2e-prepare를 완료하세요.'
        )

conversion_signature = {
    'conversion_fingerprint': CONVERSION_FINGERPRINT,
    'source_directory': str(JULIET_SOURCE.resolve()),
    'seed': SEED,
    'max_cases_per_family': JULIET_CASES_PER_FAMILY,
    'max_cases_per_cwe': JULIET_CASES_PER_CWE,
    'max_cases_per_template': JULIET_CASES_PER_TEMPLATE,
}
juliet_outputs = [JULIET_DIR / 'split_manifest.json'] + [
    JULIET_DIR / f'cases_{split}.jsonl' for split in required_splits
]
conversion_cache_matches = _read_cache(JULIET_CONVERSION_CACHE) == conversion_signature
juliet_built = (
    REBUILD_JULIET
    or not conversion_cache_matches
    or not all(path.is_file() for path in juliet_outputs)
)
if juliet_built:
    print('[1/3] Converting Juliet because data, settings, or converter code changed')
    juliet_manifest = prepare_juliet_dataset(
        JULIET_SOURCE,
        JULIET_DIR,
        seed=SEED,
        max_cases_per_family=JULIET_CASES_PER_FAMILY,
        max_cases_per_cwe=JULIET_CASES_PER_CWE,
        max_cases_per_template=JULIET_CASES_PER_TEMPLATE,
        progress=print,
    )
    JULIET_CONVERSION_CACHE.write_text(
        json.dumps(conversion_signature, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
else:
    print('[1/3] Reusing data/juliet')
    juliet_manifest = json.loads((JULIET_DIR / 'split_manifest.json').read_text(encoding='utf-8'))

feature_signature = {
    'analysis_fingerprint': ANALYSIS_FINGERPRINT,
    'conversion_signature': conversion_signature,
    'feature_schema': 'semantic-cwe-v2',
}
juliet_feature_outputs = [
    JULIET_FEATURE_DIR / 'semantic' / f'router_{split}.jsonl'
    for split in required_splits
]
feature_cache_matches = _read_cache(JULIET_FEATURE_CACHE) == feature_signature
juliet_features_built = (
    REBUILD_JULIET_FEATURES
    or juliet_built
    or not feature_cache_matches
    or not all(path.is_file() for path in juliet_feature_outputs)
)
if juliet_features_built:
    print('[2/3] Running semantic analysis on Juliet only')
    preparation_summary = prepare_phase2e_frozen_jsonl(
        JULIET_DIR,
        config=Phase2EConfig(
            seed=SEED,
            data_directory=JULIET_FEATURE_DIR,
            analysis_checkpoint_every_cases=100,
            resume_analysis=not (
                REBUILD_JULIET_FEATURES or juliet_built or not feature_cache_matches
            ),
        ),
        backends=('semantic',),
        progress=print,
    )
    JULIET_FEATURE_CACHE.write_text(
        json.dumps(feature_signature, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
else:
    print('[2/3] Reusing data/phase2e_juliet/semantic')
    preparation_summary = json.loads((JULIET_FEATURE_DIR / 'preparation_summary.json').read_text(encoding='utf-8'))

print('[3/3] Merging compact ARVO + Juliet Router JSONL')
router_manifest = merge_router_split_directories(
    [ARVO_FEATURE_DIR, JULIET_FEATURE_DIR / 'semantic'],
    DATA_DIR / 'semantic',
)

print('Juliet cases by family:')
pprint(juliet_manifest['family_case_distribution'])
print('Combined Router samples by split:')
pprint({name: item['sample_count'] for name, item in router_manifest['splits'].items()})

[1/3] Converting Juliet because data, settings, or converter code changed
Juliet selected: 100 cases; concurrency_toctou=15, integer_size_type=40, taint_api_contract=45
Juliet selected: 200 cases; concurrency_toctou=30, integer_size_type=80, taint_api_contract=90
Juliet scan: 2000/41470 supported files, 250 balanced cases selected
Juliet scan: 4000/41470 supported files, 250 balanced cases selected
Juliet scan: 6000/41470 supported files, 250 balanced cases selected
Juliet scan: 8000/41470 supported files, 250 balanced cases selected
Juliet scan: 10000/41470 supported files, 250 balanced cases selected
Juliet scan: 12000/41470 supported files, 250 balanced cases selected
Juliet scan: 14000/41470 supported files, 250 balanced cases selected
Juliet scan: 16000/41470 supported files, 250 balanced cases selected
Juliet scan: 18000/41470 supported files, 250 balanced cases selected
Juliet scan: 20000/41470 supported files, 250 balanced cases selected
Juliet scan: 22000/41470 supported files

## Router 표본 검증

학습 전에 semantic-cwe-v2 스키마, 프로젝트 분리, 그리고 E1~E6 label의 존재를 검사합니다. 여기서 실패하면 불완전한 데이터를 학습하지 않습니다.

In [20]:
train_samples = load_router_samples_jsonl(DATA_DIR / 'semantic' / 'router_train.jsonl')
dev_samples = load_router_samples_jsonl(DATA_DIR / 'semantic' / 'router_dev.jsonl')
schemas = {row.candidate.feature_schema_version for row in [*train_samples, *dev_samples]}
if schemas != {'semantic-cwe-v2'}:
    raise RuntimeError(f'Expected semantic-cwe-v2, got {sorted(schemas)}')
train_projects = {row.candidate.project_id for row in train_samples}
dev_projects = {row.candidate.project_id for row in dev_samples}
overlap = train_projects & dev_projects
if overlap:
    raise RuntimeError(f'Project leakage detected: {sorted(overlap)[:10]}')

train_distribution = Counter(label.value for row in train_samples for label in row.labels)
dev_distribution = Counter(label.value for row in dev_samples for label in row.labels)
required_families = {family.value for family in ExpertFamily}
for split_name, distribution in [('train', train_distribution), ('dev', dev_distribution)]:
    missing = required_families - set(distribution)
    if missing:
        raise RuntimeError(f'{split_name} Router samples lack families: {sorted(missing)}')
print('train/dev samples:', len(train_samples), len(dev_samples))
print('train labels:', dict(sorted(train_distribution.items())))
print('dev labels:', dict(sorted(dev_distribution.items())))

train/dev samples: 2664 453
train labels: {'concurrency_toctou': 1, 'control_state_error': 884, 'integer_size_type': 40, 'lifetime_resource': 233, 'memory_bounds': 1446, 'taint_api_contract': 60}
dev labels: {'concurrency_toctou': 10, 'control_state_error': 127, 'integer_size_type': 10, 'lifetime_resource': 61, 'memory_bounds': 225, 'taint_api_contract': 20}


## Anchor + Rare Trigger Router 학습

E1 memory와 E5 control은 항상 실행하는 Anchor로 두고, E2 lifetime, E3 integer, E4 taint, E6 concurrency는 각각 독립적인 binary trigger로 학습합니다. dev에서 rare recall 95%를 만족하는 threshold를 선택합니다.

In [21]:
anchor_router = AnchorRareRouter.fit(train_samples, seed=SEED)
anchor_calibration = anchor_router.calibrate_threshold(
    dev_samples, target_rare_recall=TARGET_RARE_RECALL
)
anchor_dev_metrics = anchor_router.evaluate(dev_samples)
anchor_router.save(ANCHOR_MODEL_PATH)
anchor_summary = {
    'artifact': str(ANCHOR_MODEL_PATH),
    'training_sources': ['ARVO v3.0.0', 'NIST Juliet C/C++ v1.3'],
    'train_samples': len(train_samples),
    'dev_samples': len(dev_samples),
    'train_label_distribution': dict(sorted(train_distribution.items())),
    'dev_label_distribution': dict(sorted(dev_distribution.items())),
    'anchors': [item.value for item in anchor_router.anchors],
    'rare_families': [item.value for item in anchor_router.rare_families],
    'calibration': to_dict(anchor_calibration),
    'dev_metrics': to_dict(anchor_dev_metrics),
}
pprint(anchor_summary)

{'anchors': ['memory_bounds', 'control_state_error'],
 'artifact': 'C:\\Users\\junhyun111\\Desktop\\llm-security\\artifacts\\phase2e\\router_anchor_rare_v2.pkl',
 'calibration': {'achieved_recall': 0.9900990099009901,
                 'rare_trigger_rate': 0.6181015452538632,
                 'target_met': True,
                 'target_recall': 0.95,
                 'threshold': 0.00022041978656305982},
 'dev_label_distribution': {'concurrency_toctou': 10,
                            'control_state_error': 127,
                            'integer_size_type': 10,
                            'lifetime_resource': 61,
                            'memory_bounds': 225,
                            'taint_api_contract': 20},
 'dev_metrics': {'average_experts_per_candidate': 4.472406181015453,
                 'exact_coverage': 0.9977924944812362,
                 'expert_coverage': 0.9977924944812362,
                 'llm_calls_saved_vs_all_six': 692,
                 'rare_precision': 0.08

## Utility Router 학습(선택)

실제 Expert 실행 결과인 outcome JSONL이 있을 때만 Utility Router를 별도로 학습합니다. Juliet의 CWE label만으로 LLM 성공률이나 비용을 꾸며내지 않습니다.

In [22]:
utility_dir = ROOT / 'data' / 'utility'
utility_train_path = utility_dir / 'outcomes_train.jsonl'
utility_dev_path = utility_dir / 'outcomes_dev.jsonl'
utility_summary = {'trained': False, 'reason': 'outcome JSONL not found'}
if utility_train_path.exists() and utility_dev_path.exists():
    utility_train = load_utility_samples_jsonl(utility_train_path)
    utility_dev = load_utility_samples_jsonl(utility_dev_path)
    assert_project_disjoint(utility_train, utility_dev, first_name='train', second_name='dev')
    gate_rows, calibration_rows = split_gate_calibration_samples(
        utility_dev, seed=SEED, gate_fraction=0.5
    )
    utility_router = BudgetedUtilityRouter.fit(
        utility_train, policy=UtilityPolicyConfig(escalation_threshold=0.85), seed=SEED
    )
    gate_candidates = utility_router.fit_escalation_gate(gate_rows, seed=SEED)
    escalation_calibration = utility_router.calibrate_threshold(
        calibration_rows, target_truth_recall=0.95
    )
    baseline_calibration = utility_router.calibrate_baselines(calibration_rows)
    utility_metrics = utility_router.evaluate(calibration_rows)
    utility_router.save(UTILITY_MODEL_PATH)
    utility_summary = {
        'trained': True,
        'artifact': str(UTILITY_MODEL_PATH),
        'train_rows': len(utility_train),
        'dev_rows': len(utility_dev),
        'gate_training_candidates': gate_candidates,
        'calibration': to_dict(escalation_calibration),
        'baseline_calibration': to_dict(baseline_calibration),
        'calibration_metrics': to_dict(utility_metrics),
    }
pprint(utility_summary)

{'reason': 'outcome JSONL not found', 'trained': False}


In [23]:
summary = {
    'seed': SEED,
    'llm_api_calls_during_training': 0,
    'router_split_manifest': router_manifest,
    'anchor_rare': anchor_summary,
    'utility': utility_summary,
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
reloaded = AnchorRareRouter.load(ANCHOR_MODEL_PATH)
if set(reloaded.rare_families) != set(anchor_router.rare_families):
    raise RuntimeError('Saved Anchor Router failed validation')
if utility_summary['trained']:
    BudgetedUtilityRouter.load(UTILITY_MODEL_PATH)
print('artifact validation: PASS')
print('model:', ANCHOR_MODEL_PATH)
print('summary:', SUMMARY_PATH)

artifact validation: PASS
model: C:\Users\junhyun111\Desktop\llm-security\artifacts\phase2e\router_anchor_rare_v2.pkl
summary: C:\Users\junhyun111\Desktop\llm-security\artifacts\phase2e\router_training_summary.json
